<a href="https://colab.research.google.com/github/hselino/YZTA-2026-Datathon/blob/main/V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

yzta_2026_datathon_path = kagglehub.competition_download('yzta-2026-datathon')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings

# Uyarıları gizle
warnings.filterwarnings('ignore')

# 1. Verileri Yükle
train = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/train.csv')
test = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/test_x.csv')

# ID ve Hedef değişkeni ayır
target_col = 'bilissel_performans_skoru'
test_ids = test['id']

y = train[target_col]
train = train.drop(['id', target_col], axis=1)
test = test.drop(['id'], axis=1)

# 2. Veri Ön İşleme (Kategorik Değişkenler)
categorical_cols = ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']

# ÇÖZÜM: Eksik (NaN) değerleri 'Bilinmiyor' metni ile doldur, sonra string ve category tiplerine çevir.
# Bu sayede CatBoost eksik değerlerde hata vermeyecektir.
for col in categorical_cols:
    train[col] = train[col].fillna('Bilinmiyor').astype(str).astype('category')
    test[col] = test[col].fillna('Bilinmiyor').astype(str).astype('category')

# 3. Model Eğitimi için K-Fold Ayarları
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Tahminleri tutacak diziler
oof_lgb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))
oof_cat = np.zeros(len(train))

test_preds_lgb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))
test_preds_cat = np.zeros(len(test))

print("Modeller Eğitiliyor...\n")

# 4. Çapraz Doğrulama (Cross Validation) Döngüsü
for fold, (train_idx, val_idx) in enumerate(kf.split(train, y)):
    print(f"--- Fold {fold + 1} ---")

    X_train, y_train = train.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = train.iloc[val_idx], y.iloc[val_idx]

    # --- LightGBM ---
    model_lgb = lgb.LGBMRegressor(
        objective='regression',
        metric='rmse',
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=7,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1
    )
    model_lgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    oof_lgb[val_idx] = model_lgb.predict(X_val)
    test_preds_lgb += model_lgb.predict(test) / n_splits

    # --- XGBoost ---
    model_xgb = xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        enable_categorical=True,
        tree_method='hist',
        random_state=42
    )
    model_xgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    oof_xgb[val_idx] = model_xgb.predict(X_val)
    test_preds_xgb += model_xgb.predict(test) / n_splits

    # --- CatBoost ---
    model_cat = CatBoostRegressor(
        iterations=1500,
        learning_rate=0.04,
        depth=6,
        eval_metric='RMSE',
        random_seed=42,
        cat_features=categorical_cols,
        verbose=False
    )
    model_cat.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100
    )
    oof_cat[val_idx] = model_cat.predict(X_val)
    test_preds_cat += model_cat.predict(test) / n_splits

# 5. Modellerin RMSE Skorlarını İncele
print("\n--- Out-of-Fold (OOF) Skorları ---")
print(f"LightGBM RMSE: {np.sqrt(mean_squared_error(y, oof_lgb)):.5f}")
print(f"XGBoost RMSE:  {np.sqrt(mean_squared_error(y, oof_xgb)):.5f}")
print(f"CatBoost RMSE: {np.sqrt(mean_squared_error(y, oof_cat)):.5f}")

# Modellerin Tahminlerini Harmanlama (Blending - %33 her biri)
oof_blend = (oof_lgb + oof_xgb + oof_cat) / 3
blend_rmse = np.sqrt(mean_squared_error(y, oof_blend))
print(f"\n---> Ensemble (Karışım) RMSE: {blend_rmse:.5f} <---")

# 6. Test Verisi İçin Nihai Tahminleri Oluştur (Blend)
final_predictions = (test_preds_lgb + test_preds_xgb + test_preds_cat) / 3

# 7. Submission Dosyasını Kaydet
submission = pd.DataFrame({
    'id': test_ids,
    'bilissel_performans_skoru': final_predictions
})

submission.to_csv('submission_ensemble.csv', index=False)
print("\n'submission_ensemble.csv' başarıyla oluşturuldu! Şimdi bu dosyayı Kaggle'a yükleyebilirsiniz.")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings

warnings.filterwarnings('ignore')

# 1. Verileri Yükle
train = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/train.csv')
test = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/test_x.csv')

target_col = 'bilissel_performans_skoru'
test_ids = test['id']
y = train[target_col]
train = train.drop(['id', target_col], axis=1)
test = test.drop(['id'], axis=1)

# ---> YENİ EKLENEN KISIM: ÖZELLİK MÜHENDİSLİĞİ (FEATURE ENGINEERING) <---
def create_features(df):
    # Uyku kalitesini belirleyen toplam iyi uyku oranı
    df['kaliteli_uyku_yuzdesi'] = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
    # Uyku düşmanlarının birleşimi (Kafein etkisi + Ekran süresi)
    df['uyku_sabotaj_skoru'] = (df['uyku_oncesi_kafein_mg'] * 0.5) + df['uyku_oncesi_ekran_suresi_dk']
    # Fiziksel aktivitenin strese oranı (Stres 0 olmasın diye +1 ekliyoruz)
    df['adim_stres_orani'] = df['gunluk_adim_sayisi'] / (df['stres_skoru'] + 1)
    # Günlük toplam yorgunluk (çalışma + uykuya dalma süresi)
    df['toplam_yorgunluk_saat'] = df['gunluk_calisma_saati'] + (df['uykuya_dalma_suresi_dk'] / 60)
    return df

train = create_features(train)
test = create_features(test)
# ------------------------------------------------------------------------

# 2. Veri Ön İşleme
categorical_cols = ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
for col in categorical_cols:
    train[col] = train[col].fillna('Bilinmiyor').astype(str).astype('category')
    test[col] = test[col].fillna('Bilinmiyor').astype(str).astype('category')

# 3. Model Eğitimi
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))
oof_cat = np.zeros(len(train))

test_preds_lgb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))
test_preds_cat = np.zeros(len(test))

print("Modeller Eğitiliyor (Yeni Özelliklerle)...\n")

for fold, (train_idx, val_idx) in enumerate(kf.split(train, y)):
    print(f"--- Fold {fold + 1} ---")
    X_train, y_train = train.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = train.iloc[val_idx], y.iloc[val_idx]

    # LightGBM (Biraz daha derinleştirildi)
    model_lgb = lgb.LGBMRegressor(
        objective='regression', metric='rmse', n_estimators=2000,
        learning_rate=0.02, max_depth=8, num_leaves=45,
        subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1
    )
    model_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_lgb[val_idx] = model_lgb.predict(X_val)
    test_preds_lgb += model_lgb.predict(test) / n_splits

    # XGBoost
    model_xgb = xgb.XGBRegressor(
        objective='reg:squarederror', n_estimators=2000,
        learning_rate=0.02, max_depth=6, subsample=0.8,
        colsample_bytree=0.8, enable_categorical=True, tree_method='hist', random_state=42
    )
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = model_xgb.predict(X_val)
    test_preds_xgb += model_xgb.predict(test) / n_splits

    # CatBoost
    model_cat = CatBoostRegressor(
        iterations=2000, learning_rate=0.03, depth=6,
        eval_metric='RMSE', random_seed=42, cat_features=categorical_cols, verbose=False
    )
    model_cat.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100)
    oof_cat[val_idx] = model_cat.predict(X_val)
    test_preds_cat += model_cat.predict(test) / n_splits

# 4. Skorlar ve AĞIRLIKLI HARMANLAMA
print("\n--- Out-of-Fold (OOF) Skorları ---")
print(f"LightGBM RMSE: {np.sqrt(mean_squared_error(y, oof_lgb)):.5f}")
print(f"XGBoost RMSE:  {np.sqrt(mean_squared_error(y, oof_xgb)):.5f}")
print(f"CatBoost RMSE: {np.sqrt(mean_squared_error(y, oof_cat)):.5f}")

# ---> YENİ: Ağırlıklı Ortalama (CatBoost'a %60, LGBM'e %30, XGB'ye %10 güven) <---
oof_blend = (oof_cat * 0.60) + (oof_lgb * 0.30) + (oof_xgb * 0.10)
blend_rmse = np.sqrt(mean_squared_error(y, oof_blend))
print(f"\n---> Ağırlıklı Ensemble RMSE: {blend_rmse:.5f} <---")

# Test Tahminleri
final_predictions = (test_preds_cat * 0.60) + (test_preds_lgb * 0.30) + (test_preds_xgb * 0.10)

submission = pd.DataFrame({'id': test_ids, 'bilissel_performans_skoru': final_predictions})
submission.to_csv('submission_weighted_ensemble.csv', index=False)
print("\n'submission_weighted_ensemble.csv' oluşturuldu! ")